# Live-demo Double-Pass Merging Chunking
Langkah-detail tiap baris (verbose) saat `SemanticDoubleMergingSplitterNodeParser` memotong teks bronkopneumonia.

In [24]:
# %% 2. Import & helper untuk verbose
import os, sys, textwrap, spacy
from llama_index.core import Document
from llama_index.core.node_parser import (
    SemanticDoubleMergingSplitterNodeParser,
    LanguageConfig,
)
from llama_index.core.node_parser.text.utils import split_by_sentence_tokenizer

In [25]:
# %% 3. Siapkan teks & parameter
text = (
    "Bronkopneumonia pada anak usia dini sering kali disebabkan oleh infeksi bakteri, virus, "
    "atau jamur yang menyerang saluran napas bagian bawah. "
    "Gejala yang umum meliputi demam tinggi, batuk berdahak, sesak napas, dan napas cepat. "
    "Pada kasus berat, dapat terjadi penurunan kesadaran dan kegagalan pernapasan. "
    "Penanganan medis yang tepat sangat penting untuk mencegah komplikasi lebih lanjut. "
    "Selain itu, pemberian asuhan gizi yang sesuai juga berperan dalam proses pemulihan pasien."
)
print("T E K S  M A S U K A N")
print(textwrap.fill(text, width=80), "\n")

T E K S  M A S U K A N
Bronkopneumonia pada anak usia dini sering kali disebabkan oleh infeksi bakteri,
virus, atau jamur yang menyerang saluran napas bagian bawah. Gejala yang umum
meliputi demam tinggi, batuk berdahak, sesak napas, dan napas cepat. Pada kasus
berat, dapat terjadi penurunan kesadaran dan kegagalan pernapasan. Penanganan
medis yang tepat sangat penting untuk mencegah komplikasi lebih lanjut. Selain
itu, pemberian asuhan gizi yang sesuai juga berperan dalam proses pemulihan
pasien. 



In [26]:
# %% 4. Sub-class yang menambahkan print di setiap keputusan
class VerboseDoublePass(SemanticDoubleMergingSplitterNodeParser):
    def _create_initial_chunks(self, sentences):
        print("--- LANGKAH _create_initial_chunks (first-pass) ---")
        initial_chunks, chunk = [], sentences[0]
        new, last_sentences = True, ""
        print(f"0. chunk = {chunk[:60]}...")
        print(f"   new = {new}")

        for idx, sent in enumerate(sentences[1:], 1):
            print(f"\n{idx}. i = {idx} ({sent[:40]}...)")
            print(f"   len(chunk)+len(sent)+1 = {len(chunk)+len(sent)+1}")

            if new:
                sim = self._sim(chunk, sent)
                print(f"   similarity(chunk, sent) = {sim:.3f}")
                if sim < self.initial_threshold and len(chunk)+len(sent)+1 <= self.max_chunk_size:
                    print("   -> simpan chunk lama, reset ke sent")
                    initial_chunks.append(chunk)
                    chunk = sent
                    continue
                chunk_sent = [chunk]
                if len(chunk)+len(sent)+1 <= self.max_chunk_size:
                    chunk_sent.append(sent)
                    chunk = self.merging_separator.join(chunk_sent)
                    last_sentences = self.merging_separator.join(chunk_sent[-2:])
                    new = False
                    print(f"   -> masuk appending, new = {new}")
                    print(f"   last_sentences = {last_sentences[:50]}...")
                else:
                    print("   -> melebihi max_chunk_size, simpan & reset")
                    initial_chunks.append(chunk)
                    chunk = sent
                    continue
            else:
                sim = self._sim(last_sentences, sent)
                print(f"   similarity(last_sentences, sent) = {sim:.3f}")
                if sim > self.appending_threshold and len(chunk)+len(sent)+1 <= self.max_chunk_size:
                    chunk_sent.append(sent)
                    last_sentences = self.merging_separator.join(chunk_sent[-2:])
                    chunk += self.merging_separator + sent
                    print(f"   -> tambahkan sent, update last_sentences")
                else:
                    print("   -> simpan chunk, reset ke sent")
                    initial_chunks.append(chunk)
                    chunk = sent
                    new = True

        initial_chunks.append(chunk)
        print(f"\nAkhir loop -> simpan sisa chunk: {chunk[:60]}...")
        return initial_chunks

    def _sim(self, a, b):
        doc_a = self.language_config.nlp(self._clean_text_advanced(a))
        doc_b = self.language_config.nlp(self._clean_text_advanced(b))
        return doc_a.similarity(doc_b)

    def _merge_initial_chunks(self, init_chunks):
        print("\n--- LANGKAH _merge_initial_chunks (second-pass) ---")
        chunks, current = [], init_chunks[0]
        print(f"0. current = {current[:60]}...")
        for i in range(1, len(init_chunks)):
            print(f"\n{i}. bandingkan current vs init_chunks[{i}]")
            sim = self._sim(current, init_chunks[i])
            print(f"   similarity = {sim:.3f}")
            if sim > self.merging_threshold and len(current)+len(init_chunks[i])+1 <= self.max_chunk_size:
                current += self.merging_separator + init_chunks[i]
                print("   -> digabung")
            else:
                print("   -> tidak digabung, simpan current")
                chunks.append(current)
                current = init_chunks[i]
        chunks.append(current)
        print(f"\nSimpan current terakhir: {current[:60]}...")
        return chunks

In [27]:
# %% 5. Jalankan verbose double-pass
config = LanguageConfig(language="english", spacy_model="en_core_web_md")
splitter = VerboseDoublePass(
    language_config=config,
    initial_threshold=0.4,
    appending_threshold=0.5,
    merging_threshold=0.5,
    max_chunk_size=512,
)

document = Document(text=text)
nodes = splitter.get_nodes_from_documents([document])
chunks = [n.get_content() for n in nodes]

--- LANGKAH _create_initial_chunks (first-pass) ---
0. chunk = Bronkopneumonia pada anak usia dini sering kali disebabkan o...
   new = True

1. i = 1 (Gejala yang umum meliputi demam tinggi, ...)
   len(chunk)+len(sent)+1 = 227
   similarity(chunk, sent) = 0.891
   -> masuk appending, new = False
   last_sentences = Bronkopneumonia pada anak usia dini sering kali di...

2. i = 2 (Pada kasus berat, dapat terjadi penuruna...)
   len(chunk)+len(sent)+1 = 305
   similarity(last_sentences, sent) = 0.941
   -> tambahkan sent, update last_sentences

3. i = 3 (Penanganan medis yang tepat sangat penti...)
   len(chunk)+len(sent)+1 = 388
   similarity(last_sentences, sent) = 0.951
   -> tambahkan sent, update last_sentences

4. i = 4 (Selain itu, pemberian asuhan gizi yang s...)
   len(chunk)+len(sent)+1 = 479
   similarity(last_sentences, sent) = 0.939
   -> tambahkan sent, update last_sentences

Akhir loop -> simpan sisa chunk: Bronkopneumonia pada anak usia dini sering kali disebabkan o...



In [28]:
# %% 6. Lihat hasil akhir
print("\n=== H A S I L  C H U N K  A K H I R ===")
for no, ch in enumerate(chunks, 1):
    print(f"\nChunk-{no} ({len(ch)} karakter):")
    print(textwrap.fill(ch, width=80))


=== H A S I L  C H U N K  A K H I R ===

Chunk-1 (479 karakter):
Bronkopneumonia pada anak usia dini sering kali disebabkan oleh infeksi bakteri,
virus, atau jamur yang menyerang saluran napas bagian bawah. Gejala yang umum
meliputi demam tinggi, batuk berdahak, sesak napas, dan napas cepat. Pada kasus
berat, dapat terjadi penurunan kesadaran dan kegagalan pernapasan. Penanganan
medis yang tepat sangat penting untuk mencegah komplikasi lebih lanjut. Selain
itu, pemberian asuhan gizi yang sesuai juga berperan dalam proses pemulihan
pasien.


In [29]:
# %% 7. Ringkasan token & karakter (spaCy)
nlp = spacy.load("en_core_web_md")   # model sudah dipakai sebelumnya

print("Ringkasan (spaCy tokenizer):")
for no, ch in enumerate(chunks, 1):
    tokens = [tok.text for tok in nlp(ch) if not tok.is_space]
    print(f"Chunk-{no}: {len(ch)} karakter, {len(tokens)} token")

Ringkasan (spaCy tokenizer):
Chunk-1: 479 karakter, 79 token
